# Chapter 2.8: Calibration & Bias in CTR Prediction

## Learning Objectives

By the end of this notebook, you will be able to:

1. Identify and explain **position bias** in CTR prediction
2. Understand **selection bias** from only observing clicks on shown items
3. Implement **Platt scaling** and **isotonic regression** for model calibration
4. Apply **Inverse Propensity Weighting (IPW)** for debiasing
5. Implement position bias correction using propensity scores
6. Understand the basics of **counterfactual learning** for CTR
7. Evaluate calibration quality with reliability diagrams

## Prerequisites

- Chapters 2.1-2.7 (CTR prediction models)
- Basic understanding of causal inference concepts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part2/chapter_2.8_calibration_bias.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/USERNAME/rec_system/main/notebooks/part2/chapter_2.8_calibration_bias.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression as SklearnLR

np.random.seed(42)
torch.manual_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Types of Bias in CTR Prediction

### Position Bias
Users are more likely to click items at **higher positions** regardless of relevance. An item at position 1 gets more clicks than the same item at position 10.

\[ P(\text{click} | \text{item}, \text{pos}) = P(\text{examine} | \text{pos}) \times P(\text{click} | \text{examine}, \text{item}) \]

### Selection Bias
We only observe clicks on **shown** items. Items the system didn't recommend have no feedback, creating a biased training signal.

### Popularity Bias
Popular items get shown more, accumulate more clicks, and get recommended even more (rich-get-richer).

> **Concept:** If we train on biased data without correction, the model learns the bias rather than true user preferences. A well-calibrated model should output \(P(\text{click})\) that matches the actual click rate across all predicted probability ranges.

## 2. Generating Biased CTR Data

In [ ]:
def generate_biased_ctr_data(n_samples=30000, n_positions=10, seed=42):
    """
    Generate CTR data with position bias.
    Click = examine(position) AND relevant(item)
    """
    rng = np.random.RandomState(seed)
    
    # Item features
    field_dims = [10, 5, 15, 20, 8]
    n_fields = len(field_dims)
    total_features = sum(field_dims)
    offsets = np.array([0] + list(np.cumsum(field_dims[:-1])))
    
    data = np.zeros((n_samples, n_fields), dtype=np.int64)
    for f in range(n_fields):
        data[:, f] = rng.randint(0, field_dims[f], n_samples)
    global_data = data + offsets[np.newaxis, :]
    
    # True relevance (without position bias)
    V = rng.randn(total_features, 4) * 0.3
    W = rng.randn(total_features) * 0.2
    
    relevance_logits = np.zeros(n_samples)
    for i in range(n_samples):
        feats = global_data[i]
        relevance_logits[i] = np.sum(W[feats])
        v_sum = np.sum(V[feats], axis=0)
        relevance_logits[i] += 0.5 * np.sum(v_sum ** 2 - np.sum(V[feats]**2, axis=0))
    
    relevance_logits -= 0.5  # Center
    true_relevance = 1.0 / (1.0 + np.exp(-relevance_logits))
    
    # Position assignment (not random -- higher relevance items tend to get better positions)
    positions = np.zeros(n_samples, dtype=np.int64)
    for i in range(n_samples):
        # Biased position assignment
        pos_probs = np.exp(-0.3 * np.arange(n_positions) + relevance_logits[i] * 0.5)
        pos_probs /= pos_probs.sum()
        positions[i] = rng.choice(n_positions, p=pos_probs)
    
    # Position bias: examination probability decreases with position
    # Using a geometric decay
    position_propensity = 1.0 / (1.0 + 0.5 * np.arange(n_positions))  # [1.0, 0.67, 0.5, ...]
    examination_prob = position_propensity[positions]
    
    # Observed click = examine AND relevant
    examined = rng.binomial(1, examination_prob)
    clicked_if_examined = rng.binomial(1, true_relevance)
    observed_clicks = examined * clicked_if_examined
    
    # True clicks (if all positions were equal) for evaluation
    true_clicks = clicked_if_examined
    
    print(f"Generated {n_samples} samples with position bias")
    print(f"Position propensity: {position_propensity[:5].round(3)}")
    print(f"Observed CTR: {observed_clicks.mean():.4f}")
    print(f"True relevance CTR: {true_clicks.mean():.4f}")
    
    # CTR by position
    for pos in range(min(5, n_positions)):
        mask = positions == pos
        if mask.sum() > 0:
            print(f"  Position {pos}: observed CTR={observed_clicks[mask].mean():.4f}, "
                  f"true relevance={true_relevance[mask].mean():.4f}, n={mask.sum()}")
    
    return (global_data, positions, observed_clicks, true_clicks, 
            true_relevance, position_propensity, field_dims)

result = generate_biased_ctr_data()
(data, positions, obs_clicks, true_clicks, true_relevance, 
 pos_propensity, field_dims) = result

total_features = sum(field_dims)
n_fields = len(field_dims)

split = 24000
X_train, X_test = data[:split], data[split:]
pos_train, pos_test = positions[:split], positions[split:]
y_train, y_test = obs_clicks[:split], obs_clicks[split:]
y_true_train, y_true_test = true_clicks[:split], true_clicks[split:]
rel_train, rel_test = true_relevance[:split], true_relevance[split:]

## 3. Naive Model (Ignoring Position Bias)

First, let's train a model that ignores position bias entirely.

In [ ]:
class SimpleCTR(nn.Module):
    def __init__(self, num_features, n_fields, embedding_dim=8, hidden_dims=None):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [64, 32]
        self.embedding = nn.Embedding(num_features, embedding_dim)
        nn.init.xavier_uniform_(self.embedding.weight)
        
        input_dim = n_fields * embedding_dim
        layers = []
        for h in hidden_dims:
            layers.extend([nn.Linear(input_dim, h), nn.ReLU(), nn.Dropout(0.1)])
            input_dim = h
        layers.append(nn.Linear(input_dim, 1))
        self.mlp = nn.Sequential(*layers)
    
    def forward(self, x):
        embed = self.embedding(x).view(x.size(0), -1)
        return self.mlp(embed).squeeze(-1)

def train_simple(model, X_train, y_train, X_test, y_test, epochs=20, lr=0.001):
    ds = TensorDataset(torch.LongTensor(X_train), torch.FloatTensor(y_train))
    loader = DataLoader(ds, batch_size=256, shuffle=True)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.BCEWithLogitsLoss()
    
    for epoch in range(epochs):
        model.train()
        for bx, by in loader:
            optimizer.zero_grad()
            criterion(model(bx), by).backward()
            optimizer.step()
    
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(torch.LongTensor(X_test))).numpy()
    return probs

# Train naive model on observed (biased) clicks
torch.manual_seed(42)
naive_model = SimpleCTR(total_features, n_fields)
naive_probs = train_simple(naive_model, X_train, y_train, X_test, y_test)

print(f"Naive model (on biased data):")
print(f"  AUC vs observed clicks: {roc_auc_score(y_test, naive_probs):.4f}")
print(f"  AUC vs true relevance:  {roc_auc_score(y_true_test, naive_probs):.4f}")

## 4. Model Calibration

A model is **well-calibrated** if: among all samples where the model predicts \(P = 0.3\), approximately 30% actually have \(y = 1\).

We measure calibration with:
- **Reliability diagram**: plot predicted probability vs actual frequency
- **Expected Calibration Error (ECE)**: \(\text{ECE} = \sum_b \frac{|B_b|}{N} |\text{acc}(B_b) - \text{conf}(B_b)|\)

In [ ]:
def reliability_diagram(y_true, y_pred, n_bins=10, ax=None, label='Model'):
    """Plot reliability diagram and return ECE."""
    if ax is None:
        fig, ax = plt.subplots()
    
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_centers = []
    bin_means = []
    bin_counts = []
    
    ece = 0.0
    for i in range(n_bins):
        mask = (y_pred >= bin_edges[i]) & (y_pred < bin_edges[i+1])
        if mask.sum() > 0:
            bin_center = y_pred[mask].mean()
            bin_mean = y_true[mask].mean()
            bin_centers.append(bin_center)
            bin_means.append(bin_mean)
            bin_counts.append(mask.sum())
            ece += mask.sum() * abs(bin_mean - bin_center)
    
    ece /= len(y_true)
    
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
    ax.plot(bin_centers, bin_means, 'o-', label=f'{label} (ECE={ece:.4f})', linewidth=2)
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    return ece

fig, ax = plt.subplots(figsize=(8, 6))
ece_naive = reliability_diagram(y_test, naive_probs, ax=ax, label='Naive Model')
ax.set_title('Reliability Diagram: Naive Model (Before Calibration)')
plt.tight_layout()
plt.show()
print(f"ECE: {ece_naive:.4f}")

## 5. Calibration Methods

### Platt Scaling
Fit a logistic regression on the model outputs:

\[ P_{\text{calibrated}} = \sigma(a \cdot f(x) + b) \]

where \(a\) and \(b\) are learned on a validation set.

### Isotonic Regression
Fit a non-parametric, monotonically increasing function that maps raw predictions to calibrated probabilities.

> **Pro Tip:** Platt scaling is better when the model is already roughly calibrated (just needs a shift/scale). Isotonic regression is more flexible but needs more validation data to avoid overfitting.

In [ ]:
# Split test set into calibration and final evaluation
cal_split = 3000
cal_probs = naive_probs[:cal_split]
cal_labels = y_test[:cal_split]
eval_probs = naive_probs[cal_split:]
eval_labels = y_test[cal_split:]
eval_true_labels = y_true_test[cal_split:]

# Platt Scaling
platt_lr = SklearnLR()
platt_lr.fit(cal_probs.reshape(-1, 1), cal_labels)
platt_calibrated = platt_lr.predict_proba(eval_probs.reshape(-1, 1))[:, 1]

# Isotonic Regression
iso_reg = IsotonicRegression(out_of_bounds='clip')
iso_reg.fit(cal_probs, cal_labels)
iso_calibrated = iso_reg.predict(eval_probs)

# Compare calibration
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

reliability_diagram(eval_labels, eval_probs, ax=axes[0], label='Uncalibrated')
axes[0].set_title('Before Calibration')

reliability_diagram(eval_labels, platt_calibrated, ax=axes[1], label='Platt Scaling')
axes[1].set_title('After Platt Scaling')

reliability_diagram(eval_labels, iso_calibrated, ax=axes[2], label='Isotonic Regression')
axes[2].set_title('After Isotonic Regression')

plt.tight_layout()
plt.show()

## 6. Position Bias Correction

To correct for position bias, we use the **examination hypothesis**:

\[ P(\text{click} | \text{item}, \text{pos}) = P(\text{examine} | \text{pos}) \times P(\text{relevant} | \text{item}) \]

The propensity score \(P(\text{examine} | \text{pos})\) can be estimated from:
1. **Randomization experiments**: randomly shuffle some results
2. **EM algorithm**: jointly estimate propensity and relevance
3. **Result-level features**: use position as a feature during training, remove at serving

In [ ]:
# Method 1: Position as a feature (PAL - Position-Aware Learning)
class PositionAwareCTR(nn.Module):
    """CTR model with explicit position modeling."""
    
    def __init__(self, num_features, n_fields, n_positions=10, embedding_dim=8):
        super().__init__()
        self.embedding = nn.Embedding(num_features, embedding_dim)
        nn.init.xavier_uniform_(self.embedding.weight)
        
        # Position embedding (separate from item features)
        self.position_embedding = nn.Embedding(n_positions, embedding_dim)
        
        # Item relevance tower
        input_dim = n_fields * embedding_dim
        self.relevance_tower = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )
        
        # Position bias tower
        self.position_tower = nn.Sequential(
            nn.Linear(embedding_dim, 16), nn.ReLU(),
            nn.Linear(16, 1)
        )
    
    def forward(self, x, position):
        embed = self.embedding(x).view(x.size(0), -1)
        pos_embed = self.position_embedding(position)
        
        relevance = self.relevance_tower(embed)
        pos_bias = self.position_tower(pos_embed)
        
        # Click = sigmoid(relevance + position_bias)
        return (relevance + pos_bias).squeeze(-1)
    
    def predict_relevance(self, x):
        """Predict relevance only (remove position bias at serving time)."""
        embed = self.embedding(x).view(x.size(0), -1)
        return self.relevance_tower(embed).squeeze(-1)

# Train position-aware model
torch.manual_seed(42)
pos_model = PositionAwareCTR(total_features, n_fields)

ds = TensorDataset(
    torch.LongTensor(X_train), torch.LongTensor(pos_train), 
    torch.FloatTensor(y_train)
)
loader = DataLoader(ds, batch_size=256, shuffle=True)
optimizer = optim.Adam(pos_model.parameters(), lr=0.001, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss()

for epoch in range(20):
    pos_model.train()
    for bx, bpos, by in loader:
        optimizer.zero_grad()
        criterion(pos_model(bx, bpos), by).backward()
        optimizer.step()

# Evaluate: use relevance-only prediction at serving time
pos_model.eval()
with torch.no_grad():
    # With position (observed)
    pos_aware_probs = torch.sigmoid(
        pos_model(torch.LongTensor(X_test), torch.LongTensor(pos_test))).numpy()
    # Without position (debiased)
    debiased_probs = torch.sigmoid(
        pos_model.predict_relevance(torch.LongTensor(X_test))).numpy()

print(f"Position-aware model:")
print(f"  AUC vs observed clicks (with pos): {roc_auc_score(y_test, pos_aware_probs):.4f}")
print(f"  AUC vs true relevance (debiased):  {roc_auc_score(y_true_test, debiased_probs):.4f}")
print(f"  AUC vs true relevance (naive):     {roc_auc_score(y_true_test, naive_probs):.4f}")

## 7. Inverse Propensity Weighting (IPW)

IPW reweights each training sample by the inverse of its propensity score to correct for position bias:

\[ \mathcal{L}_{\text{IPW}} = -\frac{1}{N} \sum_{i=1}^{N} \frac{1}{P(\text{examine} | \text{pos}_i)} \cdot \ell(\hat{y}_i, y_i) \]

Items at lower positions (with lower examination probability) get **higher weight** to compensate for fewer observed clicks.

> **Common Pitfall:** IPW can have high variance when propensity scores are very small. Use clipping: \(\max(p, \epsilon)\) to bound the weights.

In [ ]:
# IPW training
torch.manual_seed(42)
ipw_model = SimpleCTR(total_features, n_fields)

# Compute IPW weights
propensity_scores = pos_propensity[pos_train]
ipw_weights = 1.0 / np.clip(propensity_scores, 0.1, 1.0)  # Clip for stability
ipw_weights = ipw_weights / ipw_weights.mean()  # Normalize

ds = TensorDataset(
    torch.LongTensor(X_train), torch.FloatTensor(y_train),
    torch.FloatTensor(ipw_weights)
)
loader = DataLoader(ds, batch_size=256, shuffle=True)
optimizer = optim.Adam(ipw_model.parameters(), lr=0.001, weight_decay=1e-5)

for epoch in range(20):
    ipw_model.train()
    for bx, by, bw in loader:
        optimizer.zero_grad()
        logits = ipw_model(bx)
        # Weighted BCE loss
        loss = nn.functional.binary_cross_entropy_with_logits(
            logits, by, weight=bw)
        loss.backward()
        optimizer.step()

ipw_model.eval()
with torch.no_grad():
    ipw_probs = torch.sigmoid(ipw_model(torch.LongTensor(X_test))).numpy()

print(f"IPW model:")
print(f"  AUC vs true relevance: {roc_auc_score(y_true_test, ipw_probs):.4f}")
print(f"  (Naive model):         {roc_auc_score(y_true_test, naive_probs):.4f}")
print(f"  (Position-aware):      {roc_auc_score(y_true_test, debiased_probs):.4f}")

## 8. Visualization: Position Bias Effect

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Position vs observed CTR vs true relevance
pos_obs_ctr = []
pos_true_rel = []
pos_propensity_plot = []
for p in range(10):
    mask = pos_test == p
    if mask.sum() > 0:
        pos_obs_ctr.append(y_test[mask].mean())
        pos_true_rel.append(true_relevance[split:][mask].mean())
        pos_propensity_plot.append(pos_propensity[p])

x_pos = range(len(pos_obs_ctr))
axes[0].plot(x_pos, pos_obs_ctr, 'r-o', label='Observed CTR', linewidth=2)
axes[0].plot(x_pos, pos_true_rel, 'b-s', label='True Relevance', linewidth=2)
axes[0].plot(x_pos, pos_propensity_plot, 'g--^', label='Examination Prob', linewidth=2)
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Probability')
axes[0].set_title('Position Bias: Observed CTR vs True Relevance')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Model comparison: prediction vs true relevance correlation
methods = ['Naive', 'IPW', 'Pos-Aware']
auc_vs_true = [
    roc_auc_score(y_true_test, naive_probs),
    roc_auc_score(y_true_test, ipw_probs),
    roc_auc_score(y_true_test, debiased_probs)
]
auc_vs_obs = [
    roc_auc_score(y_test, naive_probs),
    roc_auc_score(y_test, ipw_probs),
    roc_auc_score(y_test, debiased_probs)
]

x = np.arange(len(methods))
width = 0.35
axes[1].bar(x - width/2, auc_vs_obs, width, label='AUC vs Observed', color='coral')
axes[1].bar(x + width/2, auc_vs_true, width, label='AUC vs True Relevance', color='steelblue')
axes[1].set_xticks(x)
axes[1].set_xticklabels(methods)
axes[1].set_ylabel('AUC')
axes[1].set_title('Debiasing Methods: AUC Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 9. Counterfactual Learning Basics

Counterfactual learning asks: **"What would have happened if we had shown a different item?"**

Key ideas:

1. **Logging policy**: The system that generated the training data (biased)
2. **Target policy**: The new system we want to evaluate (unbiased)
3. **IPS estimator**: \(\hat{V}(\pi) = \frac{1}{N} \sum_i \frac{\pi(a_i|x_i)}{\mu(a_i|x_i)} r_i\)

where \(\mu\) is the logging policy and \(\pi\) is the target policy.

> **Concept:** Counterfactual evaluation lets us estimate how a new ranking would perform using only logged data from the old ranking, without running an A/B test.

In [ ]:
# Simple counterfactual evaluation demo
# Simulate: what if we re-ranked items by true relevance vs by biased model?

def evaluate_ranking_policy(scores, true_relevance, n_positions=5):
    """
    Evaluate a ranking policy by the average true relevance of top-k items.
    (Simplified: we treat each test sample independently)
    """
    # Sort by scores (descending)
    sorted_idx = np.argsort(-scores)
    top_k_relevance = true_relevance[sorted_idx[:n_positions]].mean()
    return top_k_relevance

# Evaluate different "policies" on test set relevance
# We compute NDCG-like metrics for each model's ranking
from sklearn.metrics import ndcg_score

# Create groups of items to rank (simulate slate ranking)
group_size = 20
n_groups = len(y_test) // group_size

naive_ndcg = []
ipw_ndcg = []
debiased_ndcg = []

for g in range(n_groups):
    start, end = g * group_size, (g + 1) * group_size
    true_rel = rel_test[start:end]
    
    naive_ndcg.append(ndcg_score([true_rel], [naive_probs[start:end]]))
    ipw_ndcg.append(ndcg_score([true_rel], [ipw_probs[start:end]]))
    debiased_ndcg.append(ndcg_score([true_rel], [debiased_probs[start:end]]))

print(f"Ranking Quality (NDCG by true relevance):")
print(f"  Naive model:          {np.mean(naive_ndcg):.4f} +/- {np.std(naive_ndcg):.4f}")
print(f"  IPW model:            {np.mean(ipw_ndcg):.4f} +/- {np.std(ipw_ndcg):.4f}")
print(f"  Position-aware model: {np.mean(debiased_ndcg):.4f} +/- {np.std(debiased_ndcg):.4f}")

## 10. Calibration After Debiasing

In [ ]:
# Compare calibration of all models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

reliability_diagram(y_test, naive_probs, ax=axes[0], label='Naive')
axes[0].set_title('Naive Model Calibration')

reliability_diagram(y_test, ipw_probs, ax=axes[1], label='IPW')
axes[1].set_title('IPW Model Calibration')

reliability_diagram(y_test, pos_aware_probs, ax=axes[2], label='Position-Aware')
axes[2].set_title('Position-Aware Model Calibration')

plt.tight_layout()
plt.show()

---
## Exercises

### Exercise 1: Implement Position Bias Correction

In [ ]:
# Exercise 1: Estimate position propensity from data (without knowing ground truth)
# TODO: Implement the EM algorithm for propensity estimation:
#   E-step: Estimate P(relevant|item) using current propensity estimates
#   M-step: Estimate P(examine|position) using current relevance estimates
# TODO: Compare estimated propensity with the true propensity
# TODO: Use estimated propensity for IPW and compare with oracle IPW

def estimate_propensity_em(positions, clicks, n_positions=10, n_iters=20):
    """Estimate position propensity using EM."""
    # YOUR CODE HERE
    propensity = np.ones(n_positions)  # Initialize
    return propensity

# YOUR CODE HERE

### Exercise 2: Calibrate a CTR Model

In [ ]:
# Exercise 2: Full calibration pipeline
# TODO: Train a DeepFM model on the biased CTR data
# TODO: Apply both Platt scaling and isotonic regression for calibration
# TODO: Plot reliability diagrams before and after calibration
# TODO: Compare ECE scores
# TODO: Does calibration help with the position bias problem?

# YOUR CODE HERE

### Exercise 3: Selection Bias Simulation

In [ ]:
# Exercise 3: Simulate and correct selection bias
# TODO: Generate a dataset where only top-K items are shown to users
#   - True relevance exists for all items
#   - But clicks are only observed for shown items
# TODO: Train a model on shown-only data (biased)
# TODO: Apply IPW with propensity = P(shown | item)
# TODO: Compare biased vs debiased model on ALL items

# YOUR CODE HERE

### Exercise 4: Doubly Robust Estimator

In [ ]:
# Exercise 4: Implement the Doubly Robust (DR) estimator
# The DR estimator combines IPW with a direct model:
#   hat(V) = (1/N) * sum[ hat(r)(x_i) + (r_i - hat(r)(x_i)) * pi(a_i|x_i) / mu(a_i|x_i) ]
# It is unbiased if EITHER the propensity model OR the reward model is correct.
# TODO: Implement DR estimator
# TODO: Compare variance of IPS vs DR estimators

# YOUR CODE HERE

## Summary

In this chapter, we covered:

1. **Position bias**: Users click higher positions more, regardless of item quality
2. **Selection bias**: We only observe feedback on shown items
3. **Calibration**: Platt scaling and isotonic regression to align predicted probabilities with actual click rates
4. **IPW (Inverse Propensity Weighting)**: Reweight samples to correct for position/selection bias
5. **Position-aware models**: Separate relevance and position effects, remove position at serving
6. **Counterfactual learning**: Evaluate new rankings using logged data from old rankings

### Key Takeaways

- Training on biased click data without correction leads to models that learn the bias
- Position bias is the most common and impactful bias in CTR prediction
- IPW is simple but can have high variance; position-aware models are more stable
- Calibration is orthogonal to debiasing -- a debiased model still needs calibration
- Counterfactual evaluation reduces the need for expensive A/B tests

### Conclusion: Part 2

This concludes Part 2 of the Recommendation Systems course. We've covered the full spectrum of CTR prediction, from logistic regression to attention-based models, multi-task learning, and bias correction. These techniques form the foundation of modern industrial recommendation systems.